# 02 — SetSeq fits (4 arms × 28 nested fits)

`exp1_ref` re-fits exp_1's SetSeq (exp_1 saved no weights); `exp2_aug` and its two extra seeds use the `target_aug` training recipe. Each fit predicts both views of its held-out matches and saves its weights. Fits are cached by code + config fingerprint; progress is appended to `results/fit_log.txt`.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp2_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_2')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
import exp2_common as C
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp2_protocol as P, exp1_net as N
import exp1_common as C1
print('device', N.DEVICE, '| config', N.TrainConfig().as_dict())

PREREG verified 154a10f7cda6821e | frozen 2026-11-08T13:06:55+00:00
device cuda | config {'window': 300, 'batch': 32, 'epochs': 14, 'coverage': 2.0, 'lr': 0.002, 'weight_decay': 0.0001, 'width': 64, 'hidden': 64, 'boundary_weight': 0.5, 'boundary_pos_weight': 3.0, 'mirror': True, 'seed': 20261108}


In [2]:
log_path = C.RESULTS / 'fit_log.txt'
def log(message, **_):
    print(message, flush=True)
    with log_path.open('a') as handle:
        handle.write(message + '\n')
t0 = time.time()
predictions = {}
for arm in P.MODEL_ARMS:
    start = time.time()
    predictions[arm] = P.ensure_predictions(arm, log=log)
    log(f'ARM DONE {arm} {time.time() - start:.0f}s')
log(f'ALL FITS DONE {time.time() - t0:.0f}s, cached fits {P.cached_fit_count()}')

ARM DONE exp1_ref 0s


ARM DONE exp2_aug 0s


FIT exp2_aug_s1 pair_J03WN1_J03WPY 48.0s


FIT exp2_aug_s1 pair_J03WN1_J03WQQ 42.9s


FIT exp2_aug_s1 pair_J03WN1_J03WR9 42.9s


FIT exp2_aug_s1 pair_J03WOH_J03WOY 43.4s


... (43 outputs left out)

FIT exp2_aug_s2 outer_J03WQQ 50.8s


FIT exp2_aug_s2 outer_J03WR9 50.4s


ARM DONE exp2_aug_s2 1260s


ALL FITS DONE 2179s, cached fits 112


## Reproduction of exp_1
The `exp1_ref` re-fit uses exp_1's code, data and seed. It should reproduce exp_1's cached original-view outer predictions; any difference is reported, and `exp1_ref` (the re-fit) is used for both views so deltas stay paired.

In [3]:
check = P.reproduction_check(predictions['exp1_ref'])
C1.write_json(C.RESULTS / 'reproduction_exp1_ref.json', check)
print(json.dumps(check, indent=1))

{
 "max_abs_probability_diff": 0.0,
 "per_match_max_abs_diff": {
  "J03WMX": 0.0,
  "J03WN1": 0.0,
  "J03WOH": 0.0,
  "J03WOY": 0.0,
  "J03WPY": 0.0,
  "J03WQQ": 0.0,
  "J03WR9": 0.0
 },
 "argmax_disagreement_share": {
  "J03WMX": 0.0,
  "J03WN1": 0.0,
  "J03WOH": 0.0,
  "J03WOY": 0.0,
  "J03WPY": 0.0,
  "J03WQQ": 0.0,
  "J03WR9": 0.0
 },
 "bit_exact": true
}
